# Gold — Patient 360

Builds `mano_adb_outlook.gold.patient_360` — one row per patient with hospital, insurance, and lab-test summary fields.

In [0]:
from pyspark.sql.functions import *

# ----- Aggregate lab results per patient -----
lab_agg = (
    spark.table("mano_adb_outlook.silver.lab_results")
    .groupBy("patient_id")
    .agg(
        count("*").alias("total_tests"),
        count(when(col("result_status") == "Abnormal", True)).alias("abnormal_test_count"),
        count(when(col("result_status") == "Critical", True)).alias("critical_test_count"),
        max("test_date").alias("last_test_date"),
    )
)

# ----- Build patient_360 -----
df_patient_360 = (
    spark.table("mano_adb_outlook.silver.patients").alias("p")
    .join(
        spark.table("mano_adb_outlook.silver.hospitals").alias("h"),
        col("p.registered_hospital_id") == col("h.hospital_id"),
        "left",
    )
    .join(
        spark.table("mano_adb_outlook.silver.insurance_providers").alias("ip"),
        col("p.insurance_provider_id") == col("ip.insurance_provider_id"),
        "left",
    )
    .join(lab_agg.alias("lr"), col("p.patient_id") == col("lr.patient_id"), "left")
    .select(
        col("p.patient_id"),
        col("p.first_name"),
        col("p.last_name"),
        col("p.date_of_birth"),
        floor(datediff(current_date(), col("p.date_of_birth")) / 365.25).cast("int").alias("age"),
        col("p.gender"),
        col("p.city"),
        col("p.state"),
        col("p.registered_hospital_id").alias("hospital_id"),
        col("h.hospital_name"),
        col("p.insurance_provider_id"),
        col("ip.provider_name").alias("insurance_provider_name"),
        coalesce(col("lr.total_tests"), lit(0)).alias("total_tests"),
        coalesce(col("lr.abnormal_test_count"), lit(0)).alias("abnormal_test_count"),
        coalesce(col("lr.critical_test_count"), lit(0)).alias("critical_test_count"),
        col("lr.last_test_date"),
    )
)

# ----- Write to gold -----
df_patient_360.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("mano_adb_outlook.gold.patient_360")

In [0]:
row_count = spark.table("mano_adb_outlook.gold.patient_360").count()
dbutils.notebook.exit(str(row_count))